# HSM Visualiser — publish project + models

This notebook publishes to the HSM API in three optional stages (paths are under `/api/…`; see [OpenAPI](https://hsm-dashboard-dev.web.app/api/openapi.json)):

1. project-level environmental COG (`POST /api/projects/{project_id}/environmental-cogs` multipart: `file`, `infer_band_definitions`)
2. optional environmental label patch (`PATCH /api/projects/{project_id}/environmental-band-definitions/labels`)
3. model packages: **`POST /api/models`** when `(species, activity)` is new for the project, else **`PUT /api/models/{model_id}`** to refresh COG / metadata / pickle (pickle stored as `serialized_model.pkl`)

Set **`HSM_BASE_URL`** to either the site origin (e.g. `https://hsm-dashboard-dev.web.app`) or the API prefix including `/api`. The notebook normalizes to the `/api` base automatically.

Credentials and config live in **`notebooks/.env.hsm`** (copy from `.env.hsm.example`). That file is **gitignored**.

---

### Workflow (independent cells)

Run **Setup** (dotenv → config → helpers) once after a kernel restart. Then run the numbered steps in order; if a step fails, fix it and re-run **that cell only** — later cells use `require_hsm(...)` to tell you what must exist first.

1. **Authenticate** — sets `session`
2. **Resolve project** — sets `project`, `project_id`
3. **Upload environmental COG** — optional (`UPLOAD_ENV_COG`)
4. **Patch band labels** — optional (`PATCH_ENV_LABELS`); refetches project from API
5. **Fetch model catalog** — sets `catalog`
6. **Upload models** — optional (`UPLOAD_MODELS`)


In [16]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pyhere import here

os.chdir(here())
load_dotenv(here() / "notebooks" / ".env.hsm")


True

In [17]:
import json
from typing import Any, Dict, Iterator, List, Tuple

import requests
import rasterio

from sdm.raster.io import translate_to_cog
from sdm.utils.hsm_metadata import model_metadata_from_package

_hsm_root = os.environ["HSM_BASE_URL"].rstrip("/")
BASE_URL = _hsm_root if _hsm_root.endswith("/api") else f"{_hsm_root}/api"
EMAIL = os.environ["HSM_EMAIL"]
PASSWORD = os.environ["HSM_PASSWORD"]

# Optional config from env; defaults keep this notebook runnable out of the box.
PROJECT_NAME = os.environ.get("HSM_PROJECT_NAME", "Yorkshire Bat HSM")
INFER_BAND_DEFINITIONS = os.environ.get("HSM_INFER_BAND_DEFINITIONS", "true")

MODELS_DIR = Path(here() / "data" / "sdm_models")
PREDICTIONS_DIR = Path(here() / "data" / "sdm_predictions")
ENV_SOURCE_PATH = Path(here() / "data" / "evs" / "evs-to-model.tif")
# Project environmental stack (EPSG:3857 COG) — uploaded as-is when PREPARE_ENV_COG is False.
ENV_COG_PATH = Path(here() / "data" / "evs" / "evs-to-model-epsg3857-cog.tif")
ENV_LABELS_PATCH_PATH = Path(
    here() / "data" / "evs" / "environmental_band_labels_patch.json"
)
UPLOAD_WORK_DIR = Path(here() / "temp" / "hsm_upload_rasters")

# Toggle stages without editing notebook logic.
UPLOAD_ENV_COG = False
PATCH_ENV_LABELS = True
UPLOAD_MODELS = True
UPLOAD_SUITABILITY_COGS = True
PREPARE_ENV_COG = False
PREPARE_SUITABILITY_COGS = True

# Set to False if your API requires plain multipart fields, not explicit JSON body.
SEND_LABELS_AS_JSON = True

UPLOAD_WORK_DIR.mkdir(parents=True, exist_ok=True)

{
    "BASE_URL": BASE_URL,
    "PROJECT_NAME": PROJECT_NAME,
    "MODELS_DIR": str(MODELS_DIR),
    "PREDICTIONS_DIR": str(PREDICTIONS_DIR),
    "ENV_SOURCE_PATH": str(ENV_SOURCE_PATH),
    "ENV_COG_PATH": str(ENV_COG_PATH),
    "ENV_LABELS_PATCH_PATH": str(ENV_LABELS_PATCH_PATH),
    "UPLOAD_WORK_DIR": str(UPLOAD_WORK_DIR),
}

{'BASE_URL': 'https://hsm-dashboard-dev.web.app/api',
 'PROJECT_NAME': 'Yorkshire Bat HSM',
 'MODELS_DIR': '/Users/work/Data Science/sheffield-bats/data/sdm_models',
 'PREDICTIONS_DIR': '/Users/work/Data Science/sheffield-bats/data/sdm_predictions',
 'ENV_SOURCE_PATH': '/Users/work/Data Science/sheffield-bats/data/evs/evs-to-model.tif',
 'ENV_COG_PATH': '/Users/work/Data Science/sheffield-bats/data/evs/evs-to-model-epsg3857-cog.tif',
 'ENV_LABELS_PATCH_PATH': '/Users/work/Data Science/sheffield-bats/data/evs/environmental_band_labels_patch.json',
 'UPLOAD_WORK_DIR': '/Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters'}

In [18]:
def training_packages(models_dir: Path) -> Iterator[Tuple[Path, Path, Dict[str, Any]]]:
    for pkg_json in sorted(models_dir.glob("*/package.json")):
        pkl = pkg_json.parent / "model.pkl"
        if not pkl.is_file():
            continue
        pkg = json.loads(pkg_json.read_text(encoding="utf-8"))
        yield pkg_json, pkl, pkg


def catalog_index(models: List[Dict[str, Any]]) -> Dict[Tuple[str, str], str]:
    out: Dict[Tuple[str, str], str] = {}
    for row in models:
        sp, ac, rid = row.get("species"), row.get("activity"), row.get("id")
        if sp is not None and ac is not None and rid is not None:
            out[(sp, ac)] = rid
    return out


def suitability_cog_candidates(slug: str, predictions_dir: Path) -> List[Path]:
    return [
        predictions_dir / f"prediction_{slug}_epsg3857_cog.tif",
        predictions_dir / f"prediction_{slug}_epsg3857-cog.tif",
        predictions_dir / f"prediction_{slug}.tif",
    ]


def resolve_suitability_cog(slug: str, predictions_dir: Path) -> Path | None:
    for p in suitability_cog_candidates(slug, predictions_dir):
        if p.is_file():
            return p
    return None


def raster_epsg(path: Path) -> int | None:
    with rasterio.open(path) as ds:
        if ds.crs is None:
            return None
        return ds.crs.to_epsg()


def ensure_epsg3857_cog(
    src_path: Path,
    upload_work_dir: Path,
    output_stem: str,
) -> Path:
    src_epsg = raster_epsg(src_path)
    if src_epsg is None:
        raise ValueError(f"Raster has no CRS: {src_path}")

    warp_path = src_path
    if src_epsg != 3857:
        import subprocess

        warp_path = upload_work_dir / f"{output_stem}_epsg3857.tif"
        cmd = [
            "gdalwarp",
            "-overwrite",
            "-t_srs",
            "EPSG:3857",
            "-r",
            "bilinear",
            "-multi",
            "-co",
            "COMPRESS=DEFLATE",
            "-co",
            "TILED=YES",
            "-co",
            "BIGTIFF=IF_NEEDED",
            str(src_path),
            str(warp_path),
        ]
        subprocess.run(cmd, check=True)

    cog_path = upload_work_dir / f"{output_stem}_epsg3857_cog.tif"
    if warp_path == cog_path and cog_path.is_file():
        return cog_path

    translate_to_cog(warp_path, cog_path, profile="deflate")
    return cog_path


def authenticated_session(base_url: str, email: str, password: str) -> requests.Session:
    """base_url should be the API prefix ending in /api (set in config cell)."""
    s = requests.Session()
    r = s.post(
        f"{base_url}/auth/token",
        json={"email": email, "password": password, "admin_only": True},
        timeout=60,
    )
    _http_error_with_body(r)
    s.headers["Authorization"] = f"Bearer {r.json()['id_token']}"
    return s


def project_by_name(
    session: requests.Session, base_url: str, project_name: str
) -> Dict[str, Any]:
    r = session.get(f"{base_url}/projects", timeout=60)
    _http_error_with_body(r)
    projects = r.json()
    for p in projects:
        if p.get("name") == project_name:
            return p
    available = [p.get("name") for p in projects]
    raise ValueError(f"Project {project_name!r} not found. Available: {available}")


def post_project_environmental_cog(
    session: requests.Session,
    base_url: str,
    project_id: str,
    env_cog_path: Path,
    infer_band_definitions: str,
) -> Dict[str, Any]:
    if not env_cog_path.is_file():
        raise FileNotFoundError(f"Missing environmental COG: {env_cog_path}")

    with env_cog_path.open("rb") as fh:
        r = session.post(
            f"{base_url}/projects/{project_id}/environmental-cogs",
            data={"infer_band_definitions": infer_band_definitions},
            files={"file": (env_cog_path.name, fh, "image/tiff")},
            timeout=300,
        )
    _http_error_with_body(r)
    return r.json()


def _http_error_with_body(r: requests.Response) -> None:
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        body = (r.text or "")[:4000]
        raise requests.HTTPError(f"{e}\nResponse body:\n{body}", response=r) from e


def filter_labels_patch_to_project_bands(
    project: Dict[str, Any], patch: Dict[str, Any]
) -> tuple[Dict[str, Any] | None, str]:
    """Drop patch keys that are not machine ``name`` values on the project (unknown → API 422)."""
    bands = project.get("environmental_band_definitions") or []
    names = {b["name"] for b in bands if isinstance(b, dict) and b.get("name")}
    if not names:
        return (
            None,
            "project has no environmental_band_definitions (upload env COG first, or PATCH full definitions).",
        )
    filtered = {k: v for k, v in patch.items() if k in names}
    if not filtered:
        sample_names = sorted(names)[:8]
        sample_keys = sorted(patch.keys())[:8]
        return None, (
            "no patch keys match project band names. "
            f"example project names: {sample_names!r}; example patch keys: {sample_keys!r}"
        )
    return filtered, ""


def patch_environmental_labels(
    session: requests.Session,
    base_url: str,
    project_id: str,
    labels_patch: Path | Dict[str, Any],
    send_as_json: bool = True,
) -> Dict[str, Any]:
    if isinstance(labels_patch, Path):
        if not labels_patch.is_file():
            raise FileNotFoundError(f"Missing labels patch JSON: {labels_patch}")
        payload = json.loads(labels_patch.read_text(encoding="utf-8"))
    else:
        payload = labels_patch

    if send_as_json:
        r = session.patch(
            f"{base_url}/projects/{project_id}/environmental-band-definitions/labels",
            json=payload,
            timeout=120,
        )
    else:
        r = session.patch(
            f"{base_url}/projects/{project_id}/environmental-band-definitions/labels",
            data=json.dumps(payload),
            headers={"Content-Type": "application/json"},
            timeout=120,
        )

    _http_error_with_body(r)
    return r.json()


def fetch_project(
    session: requests.Session, base_url: str, project_id: str
) -> Dict[str, Any]:
    r = session.get(f"{base_url}/projects/{project_id}", timeout=60)
    _http_error_with_body(r)
    return r.json()


def require_hsm(*names: str) -> None:
    import inspect

    frame = inspect.currentframe()
    try:
        caller = frame.f_back if frame is not None else None
        ns = caller.f_globals if caller is not None else {}
    finally:
        del frame

    missing = [n for n in names if n not in ns]
    if not missing:
        return
    raise RuntimeError(
        "Run earlier notebook cells first (see intro workflow). Missing: "
        + ", ".join(repr(n) for n in missing)
    )

### Authenticate

Sets `session`. Requires config (`BASE_URL`, `EMAIL`, `PASSWORD`).


In [19]:
require_hsm("BASE_URL", "EMAIL", "PASSWORD")
session = authenticated_session(BASE_URL, EMAIL, PASSWORD)
print("Authenticated — `session` is ready for the next cells.")

Authenticated — `session` is ready for the next cells.


### Resolve project

Sets `project` and `project_id` from `PROJECT_NAME`. Run after **Authenticate**.

In [ ]:
require_hsm("session")
project = project_by_name(session, BASE_URL, PROJECT_NAME)
project_id = project["id"]
print(f"Using project: {project['name']} ({project_id})")

Using project: Yorkshire Bat HSM (fa9c113d-f596-479d-bf2d-1ccd68c1e7ec)


### Upload environmental COG

`POST …/environmental-cogs`. Controlled by `UPLOAD_ENV_COG` and `PREPARE_ENV_COG`.

In [ ]:
require_hsm("session", "project_id")
project_after_env = None
if UPLOAD_ENV_COG:
    env_upload_path = ENV_COG_PATH
    if PREPARE_ENV_COG:
        if ENV_COG_PATH.is_file():
            env_source_for_prep = ENV_COG_PATH
        else:
            env_source_for_prep = ENV_SOURCE_PATH
        env_upload_path = ensure_epsg3857_cog(
            env_source_for_prep,
            UPLOAD_WORK_DIR,
            "environmental_stack",
        )
    project_after_env = post_project_environmental_cog(
        session,
        BASE_URL,
        project_id,
        env_upload_path,
        INFER_BAND_DEFINITIONS,
    )
    band_count = len(project_after_env.get("environmental_band_definitions") or [])
    print(f"Uploaded environmental COG {env_upload_path.name}. bands={band_count}")
else:
    print("Skipped environmental COG upload")

Skipped environmental COG upload


### Patch environmental band labels

Uses `PATCH …/environmental-band-definitions/labels`. Refetches the project from the API first so band names match after an env COG upload. Toggle with `PATCH_ENV_LABELS`.

In [ ]:
require_hsm("session", "project_id")
if PATCH_ENV_LABELS:
    if not ENV_LABELS_PATCH_PATH.is_file():
        print(
            f"Skipped environmental labels patch: missing file {ENV_LABELS_PATCH_PATH}"
        )
    else:
        project = fetch_project(session, BASE_URL, project_id)
        if not (
            project.get("driver_artifact_root") and project.get("driver_cog_path")
        ):
            print(
                "Skipped environmental labels patch: no env COG on server yet "
                "(complete **Upload environmental COG** first; API 422 ENV_COG_NOT_ON_DISK otherwise)."
            )
        else:
            patch_raw = json.loads(ENV_LABELS_PATCH_PATH.read_text(encoding="utf-8"))
            to_send, skip_reason = filter_labels_patch_to_project_bands(project, patch_raw)
            dropped = len(patch_raw) - len(to_send or {})
            if to_send is None:
                print(f"Skipped environmental labels patch: {skip_reason}")
            else:
                if dropped:
                    print(
                        f"Labels patch: sending {len(to_send)} keys ({dropped} omitted — not on project)."
                    )
                patch_resp = patch_environmental_labels(
                    session,
                    BASE_URL,
                    project_id,
                    to_send,
                    send_as_json=SEND_LABELS_AS_JSON,
                )
                print("Patched environmental band labels", type(patch_resp).__name__)
else:
    print("Skipped environmental labels patch")

Patched environmental band labels dict


### Fetch model catalog

Builds `catalog`: `(species, activity) → model_id` for `project_id`. Upload models uses this for POST vs PUT. Re-run after creating rows.


In [ ]:
require_hsm("session", "project_id")
r = session.get(
    f"{BASE_URL}/models",
    params={"project_id": project_id},
    timeout=60,
)
r.raise_for_status()
catalog = catalog_index(r.json())
print(f"Catalog: {len(catalog)} models for this project")
len(catalog), project_id

Catalog: 0 models for this project


(0, 'fa9c113d-f596-479d-bf2d-1ccd68c1e7ec')

### Upload models

Needs `session`, `project_id`, and `catalog`. **POST** creates a missing species+activity; **PUT** updates an existing row. Re-run **Fetch model catalog** after bulk creates. Toggle `UPLOAD_MODELS` / `UPLOAD_SUITABILITY_COGS`.


In [ ]:
require_hsm("session", "project_id", "catalog")
if not UPLOAD_MODELS:
    print("Skipped model uploads")
else:
    missing_cogs: list[tuple[str, str]] = []
    failed: list[tuple[str, str]] = []
    created = 0
    updated = 0

    for _pkg_json, pkl, pkg in training_packages(MODELS_DIR):
        latin = pkg.get("latin_name") or ""
        activity = pkg.get("activity_type") or ""
        slug = pkg.get("model_id") or pkl.parent.name
        model_id = catalog.get((latin, activity))

        meta = model_metadata_from_package(pkg)
        files: dict[str, tuple[str, bytes, str]] = {
            "serialized_model_file": (
                "serialized_model.pkl",
                pkl.read_bytes(),
                "application/octet-stream",
            )
        }

        if UPLOAD_SUITABILITY_COGS:
            suitability_src = resolve_suitability_cog(slug, PREDICTIONS_DIR)
            if suitability_src is None:
                missing_cogs.append((slug, str(PREDICTIONS_DIR)))
                continue

            suitability_cog = suitability_src
            if PREPARE_SUITABILITY_COGS:
                suitability_cog = ensure_epsg3857_cog(
                    suitability_src,
                    UPLOAD_WORK_DIR,
                    f"prediction_{slug}",
                )

            files["file"] = (
                suitability_cog.name,
                suitability_cog.read_bytes(),
                "image/tiff",
            )

        if model_id:
            data = {"metadata": json.dumps(meta)}
            pr = session.put(
                f"{BASE_URL}/models/{model_id}",
                data=data,
                files=files,
                timeout=300,
            )
            if pr.status_code != 200:
                failed.append((slug, f"HTTP {pr.status_code} {pr.text[:500]}"))
                continue
            print(f"UPDATED {latin} — {activity} -> {model_id}")
            updated += 1
        else:
            if not UPLOAD_SUITABILITY_COGS or "file" not in files:
                failed.append(
                    (
                        slug,
                        "no catalog row: create needs suitability COG (UPLOAD_SUITABILITY_COGS=True and raster on disk)",
                    )
                )
                continue
            data = {
                "project_id": project_id,
                "species": latin,
                "activity": activity,
                "metadata": json.dumps(meta),
            }
            pr = session.post(
                f"{BASE_URL}/models",
                data=data,
                files=files,
                timeout=300,
            )
            if pr.status_code == 409:
                failed.append(
                    (
                        slug,
                        f"HTTP 409 row already exists — re-run **Fetch model catalog** then retry. {pr.text[:400]}",
                    )
                )
                continue
            if pr.status_code not in (200, 201):
                failed.append((slug, f"HTTP {pr.status_code} {pr.text[:500]}"))
                continue
            new_id = pr.json().get("id", "?")
            print(f"CREATED {latin} — {activity} -> {new_id}")
            created += 1

    for slug, preds_dir in missing_cogs:
        print(f"MISSING SUITABILITY COG {slug} under {preds_dir}")
    for slug, err in failed:
        print(f"FAILED {slug}: {err}")

    print(
        f"Done: {created} created, {updated} updated, "
        f"{len(missing_cogs)} missing COG, {len(failed)} errors"
    )



Creating output file that is 1961P x 1570L.
Using internal nodata values (e.g. -9999) for image /Users/work/Data Science/sheffield-bats/data/sdm_predictions/prediction_myotis_brandtii_roost.tif.
Copying nodata values from source /Users/work/Data Science/sheffield-bats/data/sdm_predictions/prediction_myotis_brandtii_roost.tif to destination /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_myotis_brandtii_roost_epsg3857.tif.
Processing prediction_myotis_brandtii_roost.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...

Reading input: /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_myotis_brandtii_roost_epsg3857.tif

Adding overviews...
Updating dataset tags...
Writing output to: /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_myotis_brandtii_roost_epsg3857_cog.tif


100 - done.
CREATED Myotis brandtii — Roost -> f216a09a-6942-481e-9084-85c165a52586
Creating output file that is 1961P x 1570L.
Using internal nodata values (e.g. -9999) for image /Users/work/Data Science/sheffield-bats/data/sdm_predictions/prediction_myotis_daubentonii_in_flight.tif.
Copying nodata values from source /Users/work/Data Science/sheffield-bats/data/sdm_predictions/prediction_myotis_daubentonii_in_flight.tif to destination /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_myotis_daubentonii_in_flight_epsg3857.tif.
Processing prediction_myotis_daubentonii_in_flight.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...

Reading input: /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_myotis_daubentonii_in_flight_epsg3857.tif

Adding overviews...
Updating dataset tags...
Writing output to: /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_myotis_daubentonii_in_flight_epsg3857_cog.tif


100 - done.
CREATED Myotis daubentonii — In flight -> 846ba14a-1481-4efc-ab1b-c7726927774c
Creating output file that is 1961P x 1570L.
Using internal nodata values (e.g. -9999) for image /Users/work/Data Science/sheffield-bats/data/sdm_predictions/prediction_myotis_daubentonii_roost.tif.
Copying nodata values from source /Users/work/Data Science/sheffield-bats/data/sdm_predictions/prediction_myotis_daubentonii_roost.tif to destination /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_myotis_daubentonii_roost_epsg3857.tif.
Processing prediction_myotis_daubentonii_roost.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...

...

100 - done.
CREATED Pipistrellus pygmaeus — Roost -> a9ee930f-8a4c-42a3-8278-834d81fc8ad2
Creating output file that is 1961P x 1570L.
Using internal nodata values (e.g. -9999) for image /Users/work/Data Science/sheffield-bats/data/sdm_predictions/prediction_plecotus_auritus_in_flight.tif.
Copying nodata values from source /Users/work/Data Science/sheffield-bats/data/sdm_predictions/prediction_plecotus_auritus_in_flight.tif to destination /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_plecotus_auritus_in_flight_epsg3857.tif.
Processing prediction_plecotus_auritus_in_flight.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...

Reading input: /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_plecotus_auritus_in_flight_epsg3857.tif

Adding overviews...
Updating dataset tags...
Writing output to: /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_plecotus_auritus_in_flight_epsg3857_cog.tif


100 - done.
CREATED Plecotus auritus — In flight -> 3809acae-7dd5-4dc8-acd3-d60f138de3f0
Creating output file that is 1961P x 1570L.
Using internal nodata values (e.g. -9999) for image /Users/work/Data Science/sheffield-bats/data/sdm_predictions/prediction_plecotus_auritus_roost.tif.
Copying nodata values from source /Users/work/Data Science/sheffield-bats/data/sdm_predictions/prediction_plecotus_auritus_roost.tif to destination /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_plecotus_auritus_roost_epsg3857.tif.
Processing prediction_plecotus_auritus_roost.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...

Reading input: /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_plecotus_auritus_roost_epsg3857.tif

Adding overviews...
Updating dataset tags...
Writing output to: /Users/work/Data Science/sheffield-bats/temp/hsm_upload_rasters/prediction_plecotus_auritus_roost_epsg3857_cog.tif


100 - done.
CREATED Plecotus auritus — Roost -> 97c9109d-d5ac-49bf-b141-3d06263ee5db
Done: 17 created, 0 updated, 0 missing COG, 0 errors
